# Recuperar a etapa 7 do notebook 18

**Use a sessão do notebook 18 que apresentou o erro.**

Copie a única célula de código abaixo para uma **nova célula após a etapa 7, antes da etapa 8**, no notebook original.
Execute **somente essa célula**. Ela autentica os dois braços já concluídos, corrige a referência dos alvos e também executa a etapa 8 para gerar o ZIP final.
Não reinicie a sessão. Não precisa executar novamente as etapas 1–7 nem preencher parâmetros.

O código não treina modelos. Se os resultados completos não puderem ser autenticados, ele interrompe e salva um diagnóstico.


In [ ]:
import hashlib,types,sys
from pathlib import Path
_FONTE_CORRECAO_18='"""Corrige a referência de contagem após o loader; preserva os treinos do 18."""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom copy import deepcopy\nimport hashlib,inspect,json\nfrom pathlib import Path\nimport numpy as np\nimport yaml\nimport ultralytics\nfrom ultralytics.cfg import get_cfg\nfrom ultralytics.data.build import build_yolo_dataset\nimport ultralytics.data.augment as augment_source\nimport diag11_core as core\nimport exp12_data as frozen_data\nfrom exp12_training import isolated_evaluation\nimport mix18_protocol as protocol\n\nVERSION=\'18-auditoria-pos-loader-1.0.1\'\n\ndef counts(values):\n    return dict(Counter(str(int(v)) for v in values))\n\ndef raw_rows(record):\n    if core.sha256_file(record[\'label\'])!=record[\'label_sha256\']:\n        raise ValueError(\'Rótulo original alterado: \'+record[\'id\'])\n    rows=np.array([list(map(float,l.split())) for l in Path(record[\'label\']).read_text(encoding=\'utf-8-sig\').splitlines() if l.strip()],dtype=np.float32).reshape(-1,5)\n    return np.unique(rows,axis=0)\n\ndef assert_pipeline(dataset):\n    # Albumentations fotométricas não afetam a contagem; uma transformação espacial\n    # externa impediria usar uma referência determinística por imagem.\n    seen=set()\n    def visit(transform):\n        if id(transform) in seen:return\n        seen.add(id(transform))\n        if type(transform).__name__==\'Albumentations\' and getattr(transform,\'contains_spatial\',False):\n            raise ValueError(\'Albumentations espacial inesperada na referência de alvos.\')\n        for child in getattr(transform,\'transforms\',[]) or []:visit(child)\n        pre=getattr(transform,\'pre_transform\',None)\n        if pre is not None:visit(pre)\n    visit(dataset.transforms)\n\ndef loader_reference(records,data,arguments,output):\n    """Repassa as imagens pelo mesmo dataset de TRAIN, sem modelo ou otimizador."""\n    output=Path(output)\n    if ultralytics.__version__!=\'8.4.150\':raise RuntimeError(\'A correção requer ultralytics==8.4.150, como o treino.\')\n    if (arguments.get(\'rect\') or arguments.get(\'workers\')!=0 or arguments.get(\'cache\')\n        or any(float(arguments.get(k,0))!=0 for k in protocol.AUGMENTATION_ZERO)):\n        raise ValueError(\'Parâmetros incompatíveis com a referência determinística do 18.\')\n    wanted={Path(r[\'image\']).resolve().as_posix():r for r in records}\n    if len(wanted)!=len(records) or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Imagens/IDs repetidos na referência.\')\n    if set(wanted)!=set(data[\'records\']):raise ValueError(\'Dataset comum diferente dos registros autenticados.\')\n    cfg=get_cfg(overrides=deepcopy(arguments))\n    with Path(data[\'yaml\']).open() as f:dataset_data=yaml.safe_load(f)\n    dataset_data[\'nc\']=len(core.NAMES)\n    names=dataset_data[\'names\']\n    names=dict(enumerate(names)) if isinstance(names,list) else {int(k):v for k,v in names.items()}\n    if names!=dict(enumerate(core.NAMES)):raise ValueError(\'Mapeamento de classes mudou no dataset comum.\')\n    rows=[];differences=[]\n    with isolated_evaluation():\n        dataset=build_yolo_dataset(cfg,dataset_data[\'train\'],cfg.batch,dataset_data,mode=\'train\',rect=False,stride=32)\n        assert_pipeline(dataset)\n        paths=[Path(f).resolve().as_posix() for f in dataset.im_files]\n        if len(paths)!=len(wanted) or set(paths)!=set(wanted):raise ValueError(\'O loader ignorou/adicionou uma imagem.\')\n        for index,key in enumerate(paths):\n            record=wanted[key];original=raw_rows(record);label=dataset.labels[index]\n            stored=np.concatenate([label[\'cls\'],label[\'bboxes\']],axis=1).astype(np.float32)\n            canon=lambda a:sorted(tuple(float(x) for x in r) for r in a)\n            if canon(original)!=canon(stored):raise ValueError(\'O cache bruto do loader diverge do rótulo autenticado: \'+record[\'id\'])\n            sample=dataset[index]\n            if Path(sample[\'im_file\']).resolve().as_posix()!=key:raise ValueError(\'O loader trocou a identidade da imagem.\')\n            raw=counts(original[:,0]);processed=counts(sample[\'cls\'].flatten().tolist())\n            if any(v>raw.get(k,0) for k,v in processed.items()):raise ValueError(\'O processamento adicionou alvos inesperados: \'+record[\'id\'])\n            row={\'id\':record[\'id\'],\'image_sha256\':record[\'sha256\'],\'label_sha256\':record[\'label_sha256\'],\n                 \'known_mask\':record[\'known_mask\'],\'raw_unique_counts\':raw,\'processed_counts\':processed}\n            rows.append(row)\n            if raw!=processed:\n                # O caso que disparou a auditoria deve ser reproduzível antes de aceitar.\n                for _ in range(2):\n                    again=dataset[index]\n                    if counts(again[\'cls\'].flatten().tolist())!=processed:\n                        raise ValueError(\'Contagem de alvos não determinística: \'+record[\'id\'])\n                differences.append({**row,\'discarded_by_class\':{k:v-processed.get(k,0) for k,v in raw.items() if v!=processed.get(k,0)}})\n            if (index+1)%250==0 or index+1==len(paths):\n                print(f\'Referência após loader: {index+1}/{len(paths)} imagens\',flush=True)\n        del dataset\n    summary={\'versao\':VERSION,\'imagens\':len(rows),\'imagens_com_alvos_filtrados\':len(differences),\n        \'alvos_filtrados_unicos\':sum(sum(r[\'discarded_by_class\'].values()) for r in differences),\n        \'ultralytics\':ultralytics.__version__,\'augment_source_sha256\':core.sha256_file(inspect.getfile(augment_source)),\n        \'reference_sha256\':protocol.digest(rows),\'argumentos\':arguments,\n        \'treino_alterado\':False,\'rotulos_alterados\':False,\'cobertura_alterada\':False,\n        \'explicacao\':\'A contagem é comparada após o mesmo pipeline TRAIN. O filtro padrão de caixas pode remover alvos pequenos/degenerados; a cobertura e os bytes originais continuam obrigatórios.\',\n        \'diferencas\':differences}\n    protocol.save_gz(output/\'referencia_alvos_pos_loader.json.gz\',rows)\n    core.save_json(output/\'diagnostico_alvos_pos_loader.json\',summary)\n    return {r[\'id\']:r for r in rows},summary\n\ndef audit_with_loader(results,plans,records,specs,data,arguments,output,frozen=None):\n    if frozen is not None:frozen_data.verify_frozen(frozen)\n    # Os registros do treino comum preservam os IDs, mas apontam para as cópias\n    # usadas de fato pelo loader. A auditoria usa essas cópias e confere os hashes.\n    common=list(data[\'records\'].values())\n    original={r[\'id\']:r for r in records}\n    if len(original)!=len(records) or len(common)!=len(records):raise ValueError(\'Manifesto de treino incompleto.\')\n    for r in common:\n        prior=original.get(r[\'id\'])\n        if prior is None or any(r[k]!=prior[k] for k in [\'sha256\',\'label_sha256\',\'known_mask\',\'source\']):\n            raise ValueError(\'Supervisão/cobertura comum alterada: \'+r[\'id\'])\n    reference,summary=loader_reference(common,data,arguments,output)\n    prior_counts=protocol.target_counts\n    try:\n        # Só muda a referência da auditoria. Nenhum loader de treino é executado\n        # pelo modelo e nenhuma checagem de máscara, gradiente ou orçamento é removida.\n        protocol.target_counts=lambda r:reference[r[\'id\']][\'processed_counts\']\n        result=protocol.audit_runs(results,plans,records,specs)\n    except Exception as exc:\n        divergences=[]\n        for arm,run in results.items():\n            for row in run[\'logs\']:\n                for j,iid in enumerate(row[\'ids\']):\n                    observed={str(k):v for k,v in row[\'gt_per_image_by_class\'][j].items()}\n                    expected=reference[iid]\n                    if observed!=expected[\'processed_counts\'] or row[\'known_masks\'][j]!=expected[\'known_mask\']:\n                        divergences.append({\'arm\':arm,\'step\':row[\'step\'],\'id\':iid,\'observed_counts\':observed,\n                            \'expected_counts\':expected[\'processed_counts\'],\'observed_mask\':row[\'known_masks\'][j],\n                            \'expected_mask\':expected[\'known_mask\']})\n        core.save_json(Path(output)/\'divergencias_nao_explicadas.json\',{\'erro\':str(exc),\'quantidade\':len(divergences),\'divergencias\':divergences})\n        raise ValueError(\'A divergência não foi explicada pelo processamento padrão; checagens preservadas. O diagnóstico foi salvo no pacote.\') from exc\n    finally:protocol.target_counts=prior_counts\n    result.update({\'auditoria_versao\':VERSION,\'alvos_comparados_pos_loader\':True,\n        \'treino_alterado_para_corrigir_auditoria\':False,\'imagens_com_alvos_filtrados\':summary[\'imagens_com_alvos_filtrados\'],\n        \'alvos_filtrados_unicos\':summary[\'alvos_filtrados_unicos\'],\'referencia_loader_sha256\':summary[\'reference_sha256\']})\n    core.save_json(Path(output)/\'auditoria_fontes.json\',result)\n    core.save_json(Path(output)/\'correcao_auditoria18.json\',{\'versao\':VERSION,\'concluida\':True,\n        \'checagens_mascara_gradiente_orcamento_preservadas\':True,\'treinamento_modificado\':False,\n        \'referencia\':\'referencia_alvos_pos_loader.json.gz\',\'diagnostico\':\'diagnostico_alvos_pos_loader.json\'})\n    print(\'Auditoria corrigida e conferida. Imagens com filtro padrão de alvos:\',summary[\'imagens_com_alvos_filtrados\'],flush=True)\n    return result\n\ndef recover(namespace,final_cell):\n    required=[\'PASTA_RESULTADOS\',\'PASTA_CODIGO\',\'BRACOS\',\'CRONOGRAMAS\',\'REGISTROS_TREINO\',\'SPECS\',\n        \'ARGUMENTOS_TREINO\',\'CONGELADOS\',\'CONTEXT_KEY\',\'INITIAL_STATE\',\'estado18\',\'registrar_falha\']\n    missing=[k for k in required if k not in namespace]\n    if missing:raise RuntimeError(\'Execute esta célula no notebook 18 original, na mesma sessão que terminou a etapa 7. Variáveis ausentes: \'+\', \'.join(missing))\n    output=Path(namespace[\'PASTA_RESULTADOS\'])\n    try:\n        results={}\n        for arm in protocol.ARMS:\n            result=namespace[\'estado18\'].load_completed(output,arm,namespace[\'CONTEXT_KEY\'],namespace[\'INITIAL_STATE\'],\n                namespace[\'SPECS\'][arm],namespace[\'CRONOGRAMAS\'][arm])\n            if result is None:raise RuntimeError(\'Não foi possível autenticar o braço completo \'+arm+\'. A recuperação foi interrompida sem iniciar outro treino.\')\n            results[arm]=result\n            print(\'Braço completo autenticado, sem repetir treino:\',arm,flush=True)\n        audit=audit_with_loader(results,namespace[\'CRONOGRAMAS\'],namespace[\'REGISTROS_TREINO\'],namespace[\'SPECS\'],\n            namespace[\'BRACOS\'][protocol.ARMS[0]],namespace[\'ARGUMENTOS_TREINO\'],output,namespace[\'CONGELADOS\'])\n        namespace[\'RESULTADOS_BRACOS\']=results;namespace[\'PAREAMENTO\']=audit\n    except Exception as exc:\n        namespace[\'registrar_falha\'](exc,\'correção da auditoria após loader\');raise\n    print(\'Gerando o relatório e o pacote final automaticamente...\',flush=True)\n    exec(compile(final_cell,\'etapa_8_original_preservada\',\'exec\'),namespace)\n'
_SHA_CORRECAO_18='bc7cd46e83265a3da81f4a35458c7ac825ee2d6a97efeca8f43938b9d7b648a6'
if hashlib.sha256(_FONTE_CORRECAO_18.encode()).hexdigest()!=_SHA_CORRECAO_18:
    raise RuntimeError('Código da correção da auditoria alterado.')
_destino_correcao=Path(PASTA_RESULTADOS)/'codigo/mix18_audit_fix.py'
_destino_correcao.parent.mkdir(parents=True,exist_ok=True)
_destino_correcao.write_text(_FONTE_CORRECAO_18,encoding='utf-8')
correcao18=types.ModuleType('mix18_audit_fix')
correcao18.__file__=str(_destino_correcao)
sys.modules['mix18_audit_fix']=correcao18
exec(compile(_FONTE_CORRECAO_18,str(_destino_correcao),'exec'),correcao18.__dict__)
core.save_json(Path(PASTA_RESULTADOS)/'codigo_correcao18.json',{
    'versao':correcao18.VERSION,'sha256':_SHA_CORRECAO_18,'contexto_treino_preservado':CONTEXT_KEY,
    'modulos_de_treino_modificados':False})

correcao18.recover(globals(),"try:\n    dados12.verify_frozen(CONGELADOS)\n    if protocolo18.digest(json.loads(REGRA_PATH.read_text()))!=protocolo18.digest(REGRA):raise RuntimeError('Regra alterada em memória.')\n    ENTRIES=[INICIAL];AVALIACOES={'candidato_10_inicial':REFERENCIAS['candidato_10']};ORCAMENTOS=[]\n    for arm in protocolo18.ARMS:\n        result=RESULTADOS_BRACOS[arm];ENTRIES+=result['entries'];AVALIACOES.update(result['evaluations']);ORCAMENTOS.append(result['budget'])\n    if len(ENTRIES)!=7:raise RuntimeError('São necessários o inicial e os seis checkpoints completos.')\n    core.save_json(PASTA_RESULTADOS/'checkpoints_avaliados.json',ENTRIES)\n    core.save_json(PASTA_RESULTADOS/'orcamentos_realizados.json',ORCAMENTOS)\n    RESUMO,SELECAO=relatorios18.report(PASTA_RESULTADOS,ENTRIES,AVALIACOES,REFERENCIAS,REGRA,PAREAMENTO,CONFIG,ORCAMENTOS)\n    if SELECAO['eligible']:\n        escolhido=SELECAO['eligible'];origem=PASTA_RESULTADOS/escolhido['weight_relative']\n        destino=PASTA_RESULTADOS/'checkpoint_final_elegivel_validacao.pt';shutil.copy2(origem,destino)\n        if core.sha256_file(destino)!=escolhido['weight_sha256']:raise RuntimeError('Cópia do peso elegível diverge.')\n    else:\n        (PASTA_RESULTADOS/'checkpoint_final_elegivel_validacao.pt').unlink(missing_ok=True)\n    core.save_json(PASTA_RESULTADOS/'integridade_final.json',{'conferida':True,'auditoria_fontes_conferida':True,\n        'supervisao_original_preservada':True,'projecao_treino_documentada':False,\n        'validacao_rotulos_alterados':False,'test_home_inferido':False,'test_oi20k_inferido':False,\n        'reserva_futura_extraida':False,'reserva_futura_inferida':False,'modelo_substituido':False})\n    erro=PASTA_RESULTADOS/'execucao_interrompida.json'\n    if erro.is_file():\n        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)\n        erro.replace(historico/('falha_resolvida_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))\n    progresso('Relatório e integridade concluídos')\n    PACOTE_FINAL=estado18.package(PASTA_RESULTADOS)\n    enviar_resultados('resultado final do experimento 18')\n    print('NOTEBOOK 18 CONCLUÍDO — dois braços pareados, avaliações e relatório salvos.')\n    print(RESUMO['mensagem'])\n    print('Arquivo de resultados:',PACOTE_FINAL)\n    display(FileLink(str(PACOTE_FINAL)))\n    display(FileLink(str(PASTA_RESULTADOS/'relatorio_experimento_18.html')))\nexcept Exception as exc:\n    registrar_falha(exc,'relatório e pacote final');raise\n")
